# N-BaIoT Preprocessing — Step 3: Prepare Labels
**Device: Danmini Doorbell**

This notebook uses the cleaned CSV from Step 2. It prepares the target column for machine learning by converting each text class label into a stable integer ID.

### Important
- The original text label is preserved in a column called `label`.
- A new integer column called `label_id` is added.
- The mapping is saved to JSON so we can interpret model predictions later.
- This step does not split the data. Train/test splitting is Step 4.


In [ ]:
from pathlib import Path
import pandas as pd
import json

# ===== CONFIGURATION =====
DEVICE = "Danmini_Doorbell"
CHUNK_SIZE = 250_000

# Find the preprocessed folder created by Step 1 / Step 2.
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
BASE_DIR = None

for root in search_roots:
    if (root / "preprocessed").is_dir():
        BASE_DIR = root
        break

if BASE_DIR is None:
    matches = [p for p in cwd.rglob("preprocessed") if p.is_dir()]
    if matches:
        BASE_DIR = matches[0].parent

if BASE_DIR is None:
    raise FileNotFoundError(
        "Could not find the 'preprocessed' folder. "
        "Run Steps 1 and 2 first."
    )

PREPROCESSED_DIR = BASE_DIR / "preprocessed"
INPUT_PATH = PREPROCESSED_DIR / f"{DEVICE}_step2_cleaned.csv"
REPORT_PATH = PREPROCESSED_DIR / f"{DEVICE}_step2_cleaning_report.json"
OUTPUT_PATH = PREPROCESSED_DIR / f"{DEVICE}_step3_labelled.csv"
MAPPING_PATH = PREPROCESSED_DIR / f"{DEVICE}_label_mapping.json"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Step 2 cleaned file not found: {INPUT_PATH}\nRun Step 2 first."
    )

print("Input file :", INPUT_PATH)
print("Output file:", OUTPUT_PATH)


---
## 3.1 Create a consistent label-to-ID mapping

We use the class names recorded by Step 2 and sort them alphabetically. This gives every class a deterministic integer ID. The same mapping must be reused when interpreting predictions.


In [ ]:
if REPORT_PATH.is_file():
    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        cleaning_report = json.load(f)
    labels_in_report = list(cleaning_report.get("labels", {}).keys())
else:
    # Fallback: discover labels from the cleaned CSV if the report is unavailable.
    labels_in_report = []

if not labels_in_report:
    discovered = set()
    for chunk in pd.read_csv(INPUT_PATH, usecols=["label"], chunksize=CHUNK_SIZE):
        discovered.update(chunk["label"].dropna().astype(str).unique())
    labels_in_report = sorted(discovered)

class_names = sorted(str(label) for label in labels_in_report)
if not class_names:
    raise ValueError("No class labels were found in the Step 2 output.")

label_to_id = {label: index for index, label in enumerate(class_names)}
id_to_label = {str(index): label for label, index in label_to_id.items()}

print(f"Number of classes: {len(class_names)}")
print("\nLabel encoding:")
for label, label_id in label_to_id.items():
    print(f"{label:22s} -> {label_id}")

with open(MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "device": DEVICE,
            "label_to_id": label_to_id,
            "id_to_label": id_to_label
        },
        f,
        indent=2
    )

print("\n✓ Mapping saved to:", MAPPING_PATH)


---
## 3.2 Add the encoded target column

We process the cleaned CSV in chunks to limit memory use. The original `label` text is retained, and `label_id` is added as an integer target column.


In [ ]:
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

rows_written = 0
encoded_counts = {label: 0 for label in class_names}
first_chunk = True

for chunk_number, chunk in enumerate(
    pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False),
    start=1
):
    if "label" not in chunk.columns:
        raise ValueError("Input chunk does not contain a 'label' column.")

    chunk["label"] = chunk["label"].astype(str)
    unknown_labels = sorted(set(chunk["label"].unique()) - set(label_to_id))
    if unknown_labels:
        raise ValueError(f"Found labels missing from mapping: {unknown_labels}")

    chunk["label_id"] = chunk["label"].map(label_to_id).astype("int64")

    counts = chunk["label"].value_counts()
    for label, count in counts.items():
        encoded_counts[label] += int(count)

    chunk.to_csv(
        OUTPUT_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )
    first_chunk = False
    rows_written += len(chunk)

    print(f"Chunk {chunk_number:>3}: encoded {len(chunk):>10,} rows")

if rows_written == 0:
    raise RuntimeError("No rows were written. Check the Step 2 cleaned dataset.")

print(f"\n✓ Encoded {rows_written:,} rows.")


---
## 3.3 Verify the encoded labels

We check that every original class has exactly one integer ID and that the total number of rows is preserved.


In [ ]:
print("=== LABEL ENCODING VERIFICATION ===")
print(f"Rows encoded : {rows_written:,}")
print(f"Classes      : {len(label_to_id)}")

print("\nClass distribution:")
for label in class_names:
    print(
        f"{label:22s} -> ID {label_to_id[label]:2d} : "
        f"{encoded_counts[label]:>12,} rows"
    )

assert sum(encoded_counts.values()) == rows_written
assert len(set(label_to_id.values())) == len(label_to_id)

# Read a small sample to verify the two columns agree.
check = pd.read_csv(OUTPUT_PATH, usecols=["label", "label_id"], nrows=20)
check["expected_id"] = check["label"].map(label_to_id)
assert (check["label_id"] == check["expected_id"]).all()

print("\n✓ Original labels and encoded IDs match.")
print("✓ No rows were removed in Step 3.")
print("✓ Mapping is saved and can be reused for model predictions.")
print("\nStep 3 complete. Ready for Step 4: Train/Test Split.")
